# Đánh giá detection và segmentation trên Penn-Fudan: AP, mAP, mIoU

Penn-Fudan Pedestrian: 170 ảnh đường phố (74 ở Fudan, 96 ở Penn), 423 người đi bộ, mỗi người một **mask**
pixel. Bạn tự viết bộ đánh giá mà mọi paper detection dùng: ghép prediction với ground truth theo IoU, đường
precision-recall, **AP** ở từng ngưỡng IoU, **mAP@[.5:.95]** kiểu COCO, và **mIoU** cho segmentation. Rồi
dùng chính bộ đánh giá đó đo YOLO pretrain trên COCO (chưa hề thấy Penn-Fudan), xem NMS, ngưỡng confidence và
nhãn thiếu ảnh hưởng tới con số thế nào.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`. IoU giữa các hộp dùng `box_iou` có sẵn bên dưới
  (thư viện, vì bạn đã tự viết ở bài Basic 03).
- Cell `explore` tải Penn-Fudan (53 MB), tải weights `yolo11n.pt` và `yolo11n-seg.pt` (khoảng 6 MB mỗi tệp, vào
  `~/.cache/ai-engineer/weights`) rồi chạy YOLO trên CPU, cả notebook khoảng 5 phút.

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải Penn-Fudan nếu chưa có (53 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

PF = ensure("penn-fudan") / "PennFudanPed"
print(PF, sorted(p.name for p in PF.iterdir()))

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision.ops import box_iou as _box_iou


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Ma trận IoU (N, M) giữa hộp xyxy a (N, 4) và b (M, 4), như bài Basic 03."""
    return _box_iou(torch.as_tensor(a, dtype=torch.float64).reshape(-1, 4),
                    torch.as_tensor(b, dtype=torch.float64).reshape(-1, 4)).numpy()

## 1. Từ mask instance tới hộp

Mỗi ảnh `PNGImages/<tên>.png` có mask `PedMasks/<tên>_mask.png`: ảnh xám, pixel 0 là nền, pixel `k > 0` thuộc
người thứ `k`.

- `instance_masks(mask) -> (K, H, W) bool`: một mask nhị phân cho mỗi giá trị khác 0, theo thứ tự giá trị tăng
  dần. Không có người thì trả shape `(0, H, W)`.
- `masks_to_boxes(masks) -> (K, 4) float64`: hộp xyxy theo **quy ước cạnh pixel**: một mask phủ cột 3, 4, 5
  có `x1 = 3`, `x2 = 6` (pixel `c` chiếm đoạn liên tục `[c, c+1)`), tương tự cho dòng. Đây là hệ toạ độ liên tục
  mà YOLO trả ra; `torchvision.ops.masks_to_boxes` dùng quy ước khác (`x2 = 5`), lệch 1 pixel.
- `load_pennfudan(root) -> list[dict]`: duyệt `PNGImages/*.png` theo tên tăng dần, mỗi ảnh một dict
  `{"name": "FudanPed00001", "image": <Path ảnh>, "boxes": <(K, 4) từ mask>}`.

In [ ]:
def instance_masks(mask: np.ndarray) -> np.ndarray:
    """mask (H, W) số nguyên, 0 là nền → (K, H, W) bool, theo id tăng dần."""
    # TODO
    raise NotImplementedError


def masks_to_boxes(masks: np.ndarray) -> np.ndarray:
    """(K, H, W) bool → (K, 4) float64 xyxy, quy ước cạnh pixel (x2 = cột phải nhất + 1)."""
    # TODO
    raise NotImplementedError


def load_pennfudan(root: Path) -> list[dict]:
    """root = .../PennFudanPed. Trả [{"name", "image", "boxes"}] theo tên ảnh tăng dần."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thống kê bộ dữ liệu, đối chiếu hộp từ mask với hộp trong Annotation/*.txt (toạ độ bắt đầu từ 1)
items = load_pennfudan(PF)
n_people = [len(it["boxes"]) for it in items]
print(f"{len(items)} ảnh, {sum(n_people)} người, {np.mean(n_people):.2f} người/ảnh, nhiều nhất {max(n_people)}")
hw = np.array([Image.open(it["image"]).size[::-1] for it in items])
print("cao", hw[:, 0].min(), "-", hw[:, 0].max(), "| rộng", hw[:, 1].min(), "-", hw[:, 1].max())
heights = np.concatenate([it["boxes"][:, 3] - it["boxes"][:, 1] for it in items])
print("chiều cao người (pixel): min", heights.min(), "trung vị", np.median(heights), "max", heights.max())

same = total = 0
for it in items:
    txt = (PF / "Annotation" / f"{it['name']}.txt").read_text()
    ann = np.array([[int(v) for v in b] for b in re.findall(r"\((\d+), (\d+)\) - \((\d+), (\d+)\)", txt)], float)
    ann[:, :2] -= 1                                    # (Xmin, Ymin) đếm từ 1, (Xmax, Ymax) là pixel cuối (bao gồm)
    total += len(ann)
    same += int(np.sum(np.all(ann == it["boxes"], axis=1)))
print(f"hộp trong Annotation trùng khít hộp từ mask (sau khi đổi toạ độ): {same}/{total}")

## 2. Ghép prediction với ground truth

`match_image(pred_boxes, pred_scores, gt_boxes, iou_thr) -> np.ndarray bool` (cùng thứ tự với
`pred_boxes`): duyệt prediction theo score **giảm dần**; mỗi prediction ghép với ground truth **chưa được
ghép** có IoU lớn nhất, nếu IoU đó $$\ge$$ `iou_thr` thì là true positive (TP) và ground truth đó bị "chiếm";
ngược lại là false positive (FP). Hai prediction trùng một người: cái score cao là TP, cái kia FP. Đây là quy
tắc ghép của COCO và Pascal VOC.

In [ ]:
def match_image(pred_boxes: np.ndarray, pred_scores: np.ndarray, gt_boxes: np.ndarray, iou_thr: float) -> np.ndarray:
    """Cờ TP (bool) cho từng prediction, cùng thứ tự với pred_boxes."""
    # TODO
    raise NotImplementedError

## 3. Precision-recall và Average Precision

Gộp prediction của **mọi ảnh**, sắp theo score giảm dần. Sau $$k$$ prediction đầu:

$$
\text{precision}_k = \frac{TP_k}{k}, \qquad \text{recall}_k = \frac{TP_k}{N_{gt}}
$$

`average_precision(tp, scores, n_gt) -> float`: AP là diện tích dưới đường precision-recall **đã làm đơn
điệu** (all-point interpolation, Pascal VOC từ 2010): thay mỗi precision bằng precision lớn nhất ở recall
$$\ge$$ nó, rồi cộng $$\sum_k (r_k - r_{k-1})\,p^{interp}_k$$ với $$r_0 = 0$$. Không có ground truth thì trả
`nan`; không có prediction thì 0.

`evaluate_detections(preds, gts, iou_thrs=IOU_THRS) -> dict`: `preds` là danh sách `(boxes, scores)` mỗi ảnh,
`gts` danh sách hộp ground truth mỗi ảnh. Với mỗi ngưỡng trong `iou_thrs` (mặc định 0.50, 0.55, …, 0.95), ghép
từng ảnh bằng `match_image` rồi tính AP trên toàn bộ. Trả `{"AP50": AP ở 0.5, "AP75": AP ở 0.75,
"AP": trung bình AP qua mọi ngưỡng}` (AP kiểu COCO, còn gọi mAP@[.5:.95]; với một lớp, mAP chính là AP).

In [ ]:
IOU_THRS = np.round(np.linspace(0.5, 0.95, 10), 2)


def average_precision(tp: np.ndarray, scores: np.ndarray, n_gt: int) -> float:
    # TODO: sắp theo score giảm dần, cộng dồn TP/FP, làm đơn điệu precision từ phải sang trái
    raise NotImplementedError


def evaluate_detections(preds: list[tuple[np.ndarray, np.ndarray]], gts: list[np.ndarray],
                        iou_thrs=IOU_THRS) -> dict[str, float]:
    # TODO
    raise NotImplementedError

## 4. mIoU cho semantic segmentation

Semantic segmentation gán **một lớp cho mỗi pixel** (ở đây 0 nền, 1 người), không phân biệt người nào.
Chuẩn đo là IoU theo lớp, tính trên **tổng pixel của cả tập** (không trung bình theo ảnh), rồi lấy trung bình
các lớp (mIoU):

$$
\mathrm{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \mathrm{mIoU} = \frac{1}{C}\sum_c \mathrm{IoU}_c
$$

- `confusion_matrix(pred, gt, num_classes) -> (C, C) int64`: `cm[i, j]` là số pixel có nhãn thật `i`, dự
  đoán `j`; `pred`, `gt` là mảng số nguyên cùng shape bất kỳ.
- `iou_from_confusion(cm) -> (per_class, miou)`: `per_class` mảng `(C,)`; lớp không xuất hiện ở cả nhãn thật
  lẫn dự đoán có IoU `nan` và bị bỏ khỏi trung bình.

In [ ]:
def confusion_matrix(pred: np.ndarray, gt: np.ndarray, num_classes: int) -> np.ndarray:
    # TODO: np.bincount trên gt * num_classes + pred
    raise NotImplementedError


def iou_from_confusion(cm: np.ndarray) -> tuple[np.ndarray, float]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: YOLO11n pretrain trên COCO (lớp 0 là person), chưa thấy Penn-Fudan; conf rất thấp vì AP cần cả đuôi PR
from ultralytics import YOLO

WEIGHTS = Path.home() / ".cache" / "ai-engineer" / "weights"
WEIGHTS.mkdir(parents=True, exist_ok=True)
det = YOLO(str(WEIGHTS / "yolo11n.pt"))        # tải 5.4 MB lần đầu

def predict_all(model, conf=0.001, iou=0.7, imgsz=640, max_det=300) -> tuple[list, float]:
    out, t0 = [], time.perf_counter()
    for it in items:
        r = model.predict(str(it["image"]), conf=conf, iou=iou, imgsz=imgsz, classes=[0], max_det=max_det,
                          verbose=False, device="cpu")[0]
        out.append((r.boxes.xyxy.numpy().astype(float), r.boxes.conf.numpy().astype(float)))
    return out, (time.perf_counter() - t0) / len(items)

gts = [it["boxes"] for it in items]
preds, sec = predict_all(det)
res = evaluate_detections(preds, gts)
print(f"yolo11n zero-shot: AP50 {res['AP50']:.3f} | AP75 {res['AP75']:.3f} | AP50-95 {res['AP']:.3f} | {sec * 1000:.0f} ms/ảnh CPU")

In [ ]:
# explore: đối chiếu với hàm AP của ultralytics (nội suy 101 điểm kiểu COCO thay vì all-point)
from ultralytics.utils.metrics import ap_per_class

scores = np.concatenate([s for _, s in preds])
tp_matrix = np.stack([np.concatenate([match_image(b, s, g, t) for (b, s), g in zip(preds, gts)]) for t in IOU_THRS], 1)
ul = ap_per_class(tp_matrix, scores, np.zeros(len(scores)), np.zeros(sum(len(g) for g in gts)))
ap_ul = ul[5][0]                                    # AP theo từng ngưỡng IoU của lớp duy nhất
print(f"ultralytics: AP50 {ap_ul[0]:.3f} | AP50-95 {ap_ul.mean():.3f}  (của bạn: {res['AP50']:.3f} | {res['AP']:.3f})")

In [ ]:
# explore: ngưỡng confidence. conf 0.25 (mặc định khi chạy app) cắt mất đuôi PR nên AP thấp hơn conf 0.001
for c in [0.001, 0.25, 0.5]:
    kept = [(b[s >= c], s[s >= c]) for b, s in preds]
    r = evaluate_detections(kept, gts)
    print(f"conf >= {c:<5}: {sum(len(s) for _, s in kept):>6} hộp | AP50 {r['AP50']:.3f} | AP50-95 {r['AP']:.3f}")

In [ ]:
# explore: NMS. Chạy YOLO với iou=1.0 (gần như tắt NMS), rồi tự áp NMS ở nhiều ngưỡng bằng torchvision
from torchvision.ops import nms

raw, _ = predict_all(det, iou=1.0, max_det=3000)
print("số hộp trước NMS:", sum(len(s) for _, s in raw), "| sau NMS mặc định (iou 0.7):", sum(len(s) for _, s in preds))
for thr in [0.3, 0.5, 0.7, 0.9, 1.0]:
    kept = []
    for b, s in raw:
        k = nms(torch.as_tensor(b), torch.as_tensor(s), thr).numpy()
        kept.append((b[k], s[k]))
    r = evaluate_detections(kept, gts)
    print(f"NMS iou {thr:.1f}: AP50 {r['AP50']:.3f} | AP50-95 {r['AP']:.3f} | {sum(len(s) for _, s in kept):>6} hộp")

In [ ]:
# explore: false positive score cao nhất: phần lớn là người thật mà Penn-Fudan không gán nhãn (bị che, ở xa)
import matplotlib.patches as patches
import matplotlib.pyplot as plt

fps = []
for it, (b, s) in zip(items, preds):
    tp = match_image(b, s, it["boxes"], 0.5)
    fps += [(float(s[i]), it, b[i]) for i in np.flatnonzero(~tp)]
fps.sort(key=lambda t: -t[0])
print("10 FP score cao nhất:", [round(f[0], 2) for f in fps[:10]])
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (score, it, box) in zip(axes, fps[:4]):
    ax.imshow(Image.open(it["image"]))
    for g in it["boxes"]:
        ax.add_patch(patches.Rectangle(g[:2], g[2] - g[0], g[3] - g[1], fill=False, ec="lime", lw=2))
    ax.add_patch(patches.Rectangle(box[:2], box[2] - box[0], box[3] - box[1], fill=False, ec="red", lw=2))
    ax.set_title(f"{it['name']} FP {score:.2f}", fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
# explore: segmentation. YOLO11n-seg zero-shot, gộp mask mọi người (conf >= 0.25) thành mask "người" rồi tính mIoU
seg = YOLO(str(WEIGHTS / "yolo11n-seg.pt"))     # tải 5.9 MB lần đầu
cm = np.zeros((2, 2), dtype=np.int64)
for it in items:
    gt = (np.asarray(Image.open(PF / "PedMasks" / f"{it['name']}_mask.png")) > 0).astype(np.int64)
    r = seg.predict(str(it["image"]), conf=0.25, classes=[0], retina_masks=True, verbose=False, device="cpu")[0]
    pred = np.zeros_like(gt)
    if r.masks is not None:
        pred = (r.masks.data.numpy().max(axis=0) > 0.5).astype(np.int64)
    cm += confusion_matrix(pred, gt, 2)
per_class, miou = iou_from_confusion(cm)
print(f"yolo11n-seg zero-shot: IoU nền {per_class[0]:.3f} | IoU người {per_class[1]:.3f} | mIoU {miou:.3f}")
print(f"pixel accuracy {np.trace(cm) / cm.sum():.3f} (cao vì nền chiếm {cm[0].sum() / cm.sum():.0%} số pixel)")

## 5. Đọc kết quả

So số của bạn với bảng ở mục 5 và 6 phần lý thuyết. Mấy câu nên tự trả lời trước khi làm trắc nghiệm: vì sao
AP50 cao mà AP50-95 thấp hơn nhiều; vì sao chạy app với conf 0.25 thì đúng còn đo AP với conf 0.25 thì sai;
false positive score cao nhất có thật là lỗi của model không; và vì sao pixel accuracy là thước đo tồi cho
segmentation khi nền chiếm phần lớn ảnh.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.